# 2C Raw Data Profiling

## tl;dr

- 정식 배치 기준 API4_3은 49,386개 출전 행·4,600경주이고 API179_1은 32,074개 승식 행·4,582경주다.
- 업무 키 중복과 필수 키 결측은 모두 0건이다.
- 매출 경주는 결과에 100% 결합되며, 결과 경주 중 18개에는 매출이 없다.
- 매출은 7개 승식이 경주마다 한 행씩 존재하고 금액 형식 오류는 0건이다.


## Context & Methods

### Key Assumptions

- Manifest에서 API별 완료 배치 중 요청 수가 가장 큰 전체 배치를 선택한다.
- API4_3 grain은 `rcDate + meet + rcNo + hrNo`, API179_1은 `rcDate + meet + rcNo + pool`이다.
- 경주 결합 키는 `rcDate + meet + rcNo`이며 경마장 명칭은 코드 1/3으로 정규화한다.
- Raw 파일은 읽기만 하며 수정하지 않는다.


In [1]:
from pprint import pprint

from kra_analytics.profiling import build_raw_profile

profile = build_raw_profile()

## Data

### 1. 배치와 기본 범위

In [2]:
summary_fields = ("files", "rows", "columns_union", "columns_common", "date_min", "date_max")
pprint(
    {
        "race_batch_id": profile["race_batch_id"],
        "sales_batch_id": profile["sales_batch_id"],
        "race": {key: profile["race"][key] for key in summary_fields},
        "sales": {key: profile["sales"][key] for key in summary_fields},
    }
)

{'race': {'columns_common': 83,
          'columns_union': 89,
          'date_max': '20260726',
          'date_min': '20240105',
          'files': 51,
          'rows': 49386},
 'race_batch_id': '20260801T152038430395Z_api4_3_34c281ed',
 'sales': {'columns_common': 6,
           'columns_union': 6,
           'date_max': '20260726',
           'date_min': '20240105',
           'files': 37,
           'rows': 32074},
 'sales_batch_id': '20260801T153512893532Z_api179_1_9d682cd1'}


## Results

### 2. 키·결측·결합 품질

In [3]:
pprint(
    {
        "race_required_missing": profile["race"]["missing_required"],
        "sales_required_missing": profile["sales"]["missing_required"],
        "race_duplicate_key_rows": profile["race"]["duplicate_business_key_rows"],
        "sales_duplicate_key_rows": profile["sales"]["duplicate_business_key_rows"],
        "race_exact_duplicates": profile["race"]["exact_duplicate_rows"],
        "sales_exact_duplicates": profile["sales"]["exact_duplicate_rows"],
        "race_distinct_races": profile["race_distinct_races"],
        "sales_distinct_races": profile["sales_distinct_races"],
        "shared_races": profile["shared_races"],
        "race_without_sales": profile["race_without_sales"],
        "sales_without_race": profile["sales_without_race"],
        "race_join_rate": profile["race_join_rate"],
        "sales_join_rate": profile["sales_join_rate"],
    }
)

{'race_distinct_races': 4600,
 'race_duplicate_key_rows': 0,
 'race_exact_duplicates': 0,
 'race_join_rate': 0.9960869565217392,
 'race_required_missing': {'hrNo': 0, 'meet': 0, 'rcDate': 0, 'rcNo': 0},
 'race_without_sales': 18,
 'sales_distinct_races': 4582,
 'sales_duplicate_key_rows': 0,
 'sales_exact_duplicates': 0,
 'sales_join_rate': 1.0,
 'sales_required_missing': {'amt': 0,
                            'meet': 0,
                            'odds': 0,
                            'pool': 0,
                            'rcDate': 0,
                            'rcNo': 0},
 'sales_without_race': 0,
 'shared_races': 4582}


### 3. 매출 도메인과 예외 경주

In [4]:
pprint(
    {
        "sales_pools": profile["sales_pools"],
        "sales_amount_invalid": profile["sales_amount_invalid"],
        "by_scope": profile["by_scope"],
    }
)
pprint(profile["race_without_sales_details"])

{'by_scope': {'2024|1': {'race_rows': 10997, 'sales_rows': 7350},
              '2024|3': {'race_rows': 7876, 'sales_rows': 5068},
              '2025|1': {'race_rows': 11020, 'sales_rows': 7294},
              '2025|3': {'race_rows': 7985, 'sales_rows': 5012},
              '2026|1': {'race_rows': 6627, 'sales_rows': 4242},
              '2026|3': {'race_rows': 4881, 'sales_rows': 3108}},
 'sales_amount_invalid': 0,
 'sales_pools': {'단식': 4582,
                 '복식': 4582,
                 '복연': 4582,
                 '삼복': 4582,
                 '삼쌍': 4582,
                 '쌍식': 4582,
                 '연식': 4582}}
[{'ord': ['99'],
  'race_key': '20240929|1|2',
  'rank': ['국6등급'],
  'rcName': ['일반'],
  'runner_rows': 11},
 {'ord': ['0', '94'],
  'race_key': '20251226|3|6',
  'rank': ['국5등급'],
  'rcName': ['일반'],
  'runner_rows': 12},
 {'ord': ['0', '1', '2', '3'],
  'race_key': '20260706|1|1',
  'rank': ['국6등급'],
  'rcName': ['일반'],
  'runner_rows': 11},
 {'ord': ['0', '1', '2', '3']

## Takeaways

1. 정식 배치 내부 업무 키 중복이 없어 후보 키를 Staging에서 유지할 수 있다.
2. API4_3의 89개 필드는 원문 중심으로 적재하며 선택 필드 결측은 그대로 보존한다.
3. API179_1의 6개 필드와 7개 승식은 안정적이며 `amt` 변환 검사를 자동화한다.
4. 매출 없는 18경주는 삭제하지 않고 품질 이슈로 격리한다.
5. 매출·확정배당은 post-race 정보이므로 예측 Feature 계층과 분리한다.
